In [0]:
dbutils.widgets.dropdown(name="env",defaultValue="dev",choices=["dev","uat","prod"],label="Select an environment:")
env=dbutils.widgets.get("env")
print(f"Running notebook for {env}")
print("*****************************")

In [0]:
checkpoint_path=spark.sql("describe external location `checkpoints`").select("url").collect()[0][0]
landing_path=spark.sql("describe external location `landing`").select("url").collect()[0][0]

In [0]:
%sql
describe external location checkpoints

In [0]:
dbutils.fs.ls("abfss://landing@testdatabricksstg.dfs.core.windows.net//raw_traffic")

In [0]:
def read_Traffic_Data():
    from pyspark.sql.types import StructType,StructField, StringType, IntegerType,FloatType,DoubleType,LongType
    from pyspark.sql.functions import current_timestamp
    schema= StructType([  StructField("Record_ID",IntegerType()),
    StructField("Count_point_id",IntegerType()),
    StructField("Direction_of_travel",StringType()),
    StructField("Year",IntegerType()),
    StructField("Count_date",StringType()),
    StructField("hour",IntegerType()),
    StructField("Region_id",IntegerType()),
    StructField("Region_name",StringType()),
    StructField("Local_authority_name",StringType()),
    StructField("Road_name",StringType()),
    StructField("Road_Category_ID",IntegerType()),
    StructField("Start_junction_road_name",StringType()),
    StructField("End_junction_road_name",StringType()),
    StructField("Latitude",DoubleType()),
    StructField("Longitude",DoubleType()),
    StructField("Link_length_km",DoubleType()),
    StructField("Pedal_cycles",IntegerType()),
    StructField("Two_wheeled_motor_vehicles",IntegerType()),
    StructField("Cars_and_taxis",IntegerType()),
    StructField("Buses_and_coaches",IntegerType()),
    StructField("LGV_Type",IntegerType()),
    StructField("HGV_Type",IntegerType()),
    StructField("EV_Car",IntegerType()),
    StructField("EV_Bike",IntegerType())])
    readTraffic_df=(spark.readStream.
                format("cloudFiles").
                option("cloudFiles.format","csv").
                option("header","true").
                option("cloudFiles.schemaLocation",f"{checkpoint_path}/rawTrafficLoad/schemaInfer").
                option("cloudFiles.inferColumnTypes","true").
                option('cloudFiles.schemaEvolutionMode','addNewColumns').
                #schema(schema).
                load(f"{landing_path}/raw_traffic"))
    
    transformed_df = readTraffic_df.withColumn("Extract_Time", current_timestamp())
    
    print("Read stream for traffic data with autoloader is successful")
    print("*********************")
    return transformed_df



In [0]:
def write_Traffic_Data(readTraffic_df,env):
    print(f'Writing data to {env}_catalog raw_traffic table')
    write_df=(readTraffic_df.writeStream.
          format('delta').
          outputMode("append").
          trigger(availableNow=True).
          option("checkpointLocation",f"{checkpoint_path}/rawTrafficLoad/checkPt").
          option("mergeSchema", "true").
          queryName("rawRoadsWriteStream").
          toTable(f"`{env}_catalog`.`bronze`.`raw_traffic`"))
    write_df.awaitTermination()
    print("Write stream for traffic data successful.")
    print("*********************")

Autoloader automatically adds _rescue_data column to the streaming data if schema is inferred using option 'cloudFiles.schemaLocation'. So either enforce static schema using schema option or set the schemaEvolutionMode to None or use option("rescuedDataColumn", "") to drop this column. But this will not accomodate schema evolution/changes, simple discard them.
So for prod-ready systems it is recommended to use option('cloudFiles.schemaEvolutionMode','addNewColumns') in readStream and for writeStream ensure to add option("mergeSchema", "true") to handle the additional columns. Note addNewColumns works for schemaEvol not for schemaRevolution as it does not entertain datatype mismatches for existing columns.

In [0]:
def read_Road_Data():
    from pyspark.sql.types import StructType,StructField, StringType, IntegerType,FloatType,DoubleType,LongType
    from pyspark.sql.functions import current_timestamp
    schema=StructType([
        StructField('Road_ID',IntegerType()),

        StructField('Road_Category_Id',IntegerType()),

        StructField('Road_Category',StringType()),

        StructField('Region_ID',IntegerType()),

        StructField('Region_Name',StringType()),

        StructField('Total_Link_Length_Km',DoubleType()),

        StructField('Total_Link_Length_Miles',DoubleType()),

        StructField('All_Motor_Vehicles',DoubleType())])
    rawRoads_stream = (spark.readStream.
                       format('cloudFiles').
                       option('cloudFiles.format','csv').
                       option("header","true").
                       option('cloudFiles.schemaLocation',f'{checkpoint_path}/rawRoadsLoad/schemaInfer').
                       option("cloudFiles.inferColumnTypes","true").
                       option('cloudFiles.schemaEvolutionMode','addNewColumns'). # use None to avoid having _rescue_data schema mismatch.
                       #schema(schema).
                       load(f'{landing_path}/raw_roads'))
    transformed_df=rawRoads_stream.withColumn("Extract_Time", current_timestamp())
    print("Read roads stream with autoloader is successful")
    print("*********************")
    return transformed_df

In [0]:
def write_Road_Data(read_roads,env):
    print(f'Writing data to {env}_catalog raw_roads table')
    write_df=(read_roads.writeStream.
          format('delta').
          outputMode("append").
          trigger(availableNow=True).
          option("checkpointLocation",f"{checkpoint_path}/rawRoadsLoad/checkPt").
          option("mergeSchema", "true").
          queryName("rawRoadsWriteStream").
          toTable(f"`{env}_catalog`.`bronze`.`raw_roads`"))
    write_df.awaitTermination()
    print("Write stream for roads data is successful.")
    print("*********************")

In [0]:
## Reading the raw_traffic's data from landing to Bronze
read_Df = read_Traffic_Data()

## Reading the raw_roads's data from landing to Bronze
read_roads = read_Road_Data()

## Writing the raw_roads's data from landing to Bronze
write_Road_Data(read_roads,env)

## Writing the raw_traffic's data from landing to Bronze
write_Traffic_Data(read_Df,env)

###### validating data load

In [0]:
display(spark.sql(f"select * from `{env}_catalog`.`bronze`.`raw_traffic`"))

In [0]:
display(spark.sql(f"select * from `{env}_catalog`.`bronze`.`raw_roads`"))

In [0]:
%sql
select count(*) from  `dev_catalog`.`bronze`.`raw_traffic`;

In [0]:
dbutils.fs.ls('abfss://checkpoints@testdatabricksstg.dfs.core.windows.net/rawRoadsLoad/schemaInfer/_schemas')

In [0]:
%sql
select * from JSON.`abfss://checkpoints@testdatabricksstg.dfs.core.windows.net/rawRoadsLoad/schemaInfer/_schemas/0`